# Notebook 02 - Alpha Sensitivity

**Goal:** Find the alpha value that minimises cumulative regret on the Kigali-calibrated simulation.

**Gate:** One alpha clearly outperforms the others. Record it in `configs/config.yaml` before building the API.

**Alpha controls exploration-exploitation:**
- Low alpha (0.1): exploits early, misses better arms if initial sample is unlucky
- High alpha (2.0): explores too long, slow to converge
- Target: alpha that converges fast AND finds the best arms

**Sweep:** alpha in {0.1, 0.5, 1.0, 1.5, 2.0} — 5 seeds each.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timedelta

from bandits.linucb import LinUCB
from features.context_builder import ContextBuilder, N_FEATURES, CATEGORIES

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})
print(f'N_FEATURES = {N_FEATURES}  |  Imports OK')

In [ ]:
rng_setup = np.random.default_rng(42)
N_ARMS, N_ROUNDS = 30, 4000
ALPHAS = [0.1, 0.5, 1.0, 1.5, 2.0]
N_SEEDS = 5

true_quality = np.concatenate([rng_setup.uniform(16, 22, size=5), rng_setup.uniform(0, 5, size=25)])
rng_setup.shuffle(true_quality)
best_arm_reward = true_quality.max()
arm_ids = [f'product_{i:03d}' for i in range(N_ARMS)]

print(f'Best arm: {best_arm_reward:.2f}  |  Sweep: {ALPHAS}  |  Seeds per alpha: {N_SEEDS}')

In [ ]:
def run_linucb(alpha, seed, noise_std=2.0):
    rng_sim = np.random.default_rng(seed)
    BASE = datetime(2026, 1, 1)
    model = LinUCB(n_features=N_FEATURES, alpha=alpha)
    instant_regret = np.zeros(N_ROUNDS)

    for t in range(N_ROUNDS):
        ts = BASE + timedelta(hours=t % 24)
        affinity = dict(zip(CATEGORIES, rng_sim.dirichlet([0.4] * 5)))
        arm_contexts = []
        for i in range(N_ARMS):
            ctx = ContextBuilder.build(
                timestamp=ts, device_type='mobile', category_affinity=affinity,
                session_depth=int(rng_sim.integers(1, 6)),
                price_tier=float(true_quality[i] / 22.0),
                product_category=CATEGORIES[i % len(CATEGORIES)],
                seller_quality_score=float(true_quality[i] / 22.0),
                days_since_listed=float(rng_sim.random()),
                seller_delivery_reliability=float(true_quality[i] / 22.0),
            )
            arm_contexts.append(ctx)

        order = rng_sim.permutation(N_ARMS)
        scored = [(arm_ids[order[k]], model.score(arm_ids[order[k]], arm_contexts[order[k]])) for k in range(N_ARMS)]
        scored.sort(key=lambda x: x[1], reverse=True)
        chosen_id = scored[0][0]
        chosen_idx = arm_ids.index(chosen_id)
        reward = float(rng_sim.normal(true_quality[chosen_idx], noise_std))
        model.log(chosen_id, arm_contexts[chosen_idx], reward)
        model.flush()
        instant_regret[t] = best_arm_reward - true_quality[chosen_idx]

    return np.cumsum(instant_regret)

print('run_linucb() ready')

In [ ]:
results = {}

for alpha in ALPHAS:
    runs = []
    for seed in range(N_SEEDS):
        cum = run_linucb(alpha, seed=seed * 17 + 100)
        runs.append(cum)
    results[alpha] = np.array(runs)
    m = results[alpha][:, -1].mean()
    s = results[alpha][:, -1].std()
    print(f'alpha={alpha}  mean final regret: {m:,.0f}  std: {s:,.0f}')

best_alpha = ALPHAS[int(np.argmin([results[a][:, -1].mean() for a in ALPHAS]))]
print(f'\nBest alpha: {best_alpha}')

In [ ]:
rounds = np.arange(1, N_ROUNDS + 1)
colors = ['#7c3aed', '#2563eb', '#16a34a', '#ea580c', '#dc2626']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for alpha, color in zip(ALPHAS, colors):
    mean_curve = results[alpha].mean(axis=0)
    std_curve  = results[alpha].std(axis=0)
    ax.plot(rounds, mean_curve, color=color, lw=2, label=f'a={alpha}')
    ax.fill_between(rounds, mean_curve - std_curve, mean_curve + std_curve, color=color, alpha=0.12)
ax.set_xlabel('Round'); ax.set_ylabel('Cumulative regret (mean +/- 1 std)')
ax.set_title('Alpha Sensitivity')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

ax = axes[1]
means = [results[a][:, -1].mean() for a in ALPHAS]
stds  = [results[a][:, -1].std()  for a in ALPHAS]
bars = ax.bar([str(a) for a in ALPHAS], means, yerr=stds, color=colors, width=0.5, capsize=4)
best_idx = int(np.argmin(means))
bars[best_idx].set_edgecolor('black'); bars[best_idx].set_linewidth(2)
ax.set_xlabel('Alpha'); ax.set_ylabel('Final cumulative regret')
ax.set_title('Final Regret by Alpha (lower = better)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('../docs/02_alpha_sensitivity.png', bbox_inches='tight')
plt.show()
print(f'Best alpha: {ALPHAS[best_idx]} -> set bandit.alpha = {ALPHAS[best_idx]} in configs/config.yaml')

In [ ]:
# Convergence speed: rounds until rolling per-round regret drops below 1.0
THRESHOLD = 1.0
window = 100
print(f'Rounds until avg per-round regret < {THRESHOLD} (rolling {window}):')
print()
for alpha in ALPHAS:
    conv = []
    for run in results[alpha]:
        ir = np.diff(np.concatenate([[0], run]))
        rolling = np.convolve(ir, np.ones(window)/window, mode='valid')
        below = np.where(rolling < THRESHOLD)[0]
        conv.append(below[0] + window if len(below) > 0 else N_ROUNDS)
    print(f'  alpha={alpha}: {np.mean(conv):.0f} rounds  (never: {sum(c == N_ROUNDS for c in conv)}/{N_SEEDS})')

## Decision

Record the best alpha from the bar chart in `configs/config.yaml`:

```yaml
bandit:
  alpha: <best_alpha>
```

This value is fixed before building the API in Phase ML-5. Re-evaluate after Phase ML-7 when real interaction data arrives.